## Colab Setup

In [1]:
import os

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/ImmunoPath"
DATA_DIR = f"{PROJECT_DIR}/data"
RESULTS_DIR = f"{PROJECT_DIR}/results/phase4"
TRAINING_DIR = f"{DATA_DIR}/training"

os.makedirs(RESULTS_DIR, exist_ok=True)

# HuggingFace login
from huggingface_hub import login
from google.colab import userdata
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Logged in to HuggingFace")
except Exception:
    print("Set HF_TOKEN in Colab Secrets")

# GPU check
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("No GPU!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Logged in to HuggingFace
GPU: NVIDIA L4 (23.7 GB)


## Install Dependencies

In [12]:
import subprocess
subprocess.run([
    "pip", "install", "-q", "--upgrade",
    "accelerate>=0.34.0",
    "pandas", "numpy", "scikit-learn", "scipy", "tqdm",
], check=True)

# Try to install flash-attention (may fail on some runtimes)
try:
    subprocess.run(["pip", "install", "-q", "flash-attn", "--no-build-isolation"], check=True)
    FLASH_ATTN_AVAILABLE = True
    print("Flash Attention 2 installed")
except Exception:
    FLASH_ATTN_AVAILABLE = False
    print("Flash Attention not available (using default attention)")


Flash Attention 2 installed


## Load MedGemma (Zero-Shot — No Adapters)

In [3]:
!pip uninstall -y pillow
!pip install --no-cache-dir pillow==10.3.0
!pip install --upgrade --no-cache-dir transformers


Found existing installation: pillow 11.3.0
Uninstalling pillow-11.3.0:
  Successfully uninstalled pillow-11.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 293.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.0 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 30.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [4]:
!pip install -U bitsandbytes>=0.46.1

In [13]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
import torch
import gc

MODEL_ID = "google/medgemma-1.5-4b-it"

print(f"Loading {MODEL_ID} (zero-shot, 4-bit quantized)...")

processor = AutoProcessor.from_pretrained(MODEL_ID)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

attn_impl = "flash_attention_2" if FLASH_ATTN_AVAILABLE else "eager"
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation=attn_impl,
    quantization_config=bnb_config,
)
model.eval()

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    if hasattr(torch.backends.cuda, 'matmul'):
        torch.backends.cuda.matmul.allow_tf32 = True
    if hasattr(torch.backends.cudnn, 'allow_tf32'):
        torch.backends.cudnn.allow_tf32 = True

allocated = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
print(f"Model loaded ({attn_impl}, 4-bit)")
print(f"   VRAM: {allocated:.2f} GB")


Loading google/medgemma-1.5-4b-it (zero-shot, 4-bit quantized)...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

Model loaded (flash_attention_2, 4-bit)
   VRAM: 3.24 GB


/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:42: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return self.getter()


##  Load Test + Val Data

In [2]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import time

def load_jsonl(path: str) -> list:
    """Load JSONL file."""
    samples = []
    with open(path) as f:
        for line in f:
            samples.append(json.loads(line.strip()))
    return samples

# Load test and val sets
test_samples = load_jsonl(f"{TRAINING_DIR}/test.jsonl") if os.path.exists(f"{TRAINING_DIR}/test.jsonl") else []
val_samples = load_jsonl(f"{TRAINING_DIR}/val.jsonl") if os.path.exists(f"{TRAINING_DIR}/val.jsonl") else []

# Combine for baseline evaluation (we want metrics on both)
eval_samples = test_samples + val_samples
print(f"Loaded {len(test_samples)} test + {len(val_samples)} val = {len(eval_samples)} eval samples")

# Also load ground truth signatures for metrics
SIGNATURES_PATH = f"{DATA_DIR}/signatures/immune_signatures.csv"
ground_truth_df = pd.read_csv(SIGNATURES_PATH, index_col=0)
print(f"Ground truth: {len(ground_truth_df)} patients")

Loaded 94 test + 94 val = 188 eval samples
Ground truth: 950 patients


## Efficient Image Loading

In [6]:
def normalize_prediction_keys(parsed_json: dict) -> dict:
    """Map variant model output keys to canonical schema keys.
    Lowercases all keys first to handle mixed-case model outputs.
    """
    if not parsed_json:
        return parsed_json
    KEY_MAP = {
        "cd274_rna_proxy_level": "cd274_expression",
        "cd274_rna_proxy": "cd274_expression",
        "cd274_proxy": "cd274_expression",
        "cd274_proxy_level": "cd274_expression",
        "cd274_level": "cd274_expression",
        "pdl1_expression": "cd274_expression",
        "overall_immune_score": "immune_score",
        "cd8_t_cell_infiltration": "cd8_infiltration",
        "tme_subtype_confidence": "prediction_entropy",
        "til_bucket": "til_density",
        "msi_probability": "msi_probability",
        "uncertainty": "prediction_entropy",
    }
    normalized = {}
    for k, v in parsed_json.items():
        k_lower = k.lower()
        canonical = KEY_MAP.get(k_lower, k_lower)
        normalized[canonical] = v
    return normalized


def load_patches_parallel(patch_paths: list, max_patches: int = 8) -> list:
    paths = patch_paths[:max_patches]
    def _load_one(path):
        try:
            return Image.open(path).convert("RGB")
        except Exception:
            return None
    with ThreadPoolExecutor(max_workers=min(4, len(paths))) as executor:
        results = list(executor.map(_load_one, paths))
    images = [img for img in results if img is not None]
    return images

if eval_samples:
    test_imgs = load_patches_parallel(eval_samples[0]["patch_paths"])
    print(f"Image loading test: {len(test_imgs)} images loaded")
    if test_imgs:
        print(f"   Size: {test_imgs[0].size}, Mode: {test_imgs[0].mode}")


Image loading test: 8 images loaded
   Size: (512, 512), Mode: RGB


## Zero-Shot Inference Function

In [15]:
def run_zero_shot_inference(
    sample: dict,
    model,
    processor,
    max_new_tokens: int = 600,
) -> dict:
    """
    Run zero-shot MedGemma inference on a single sample.
    Returns the raw model output + parsed JSON (if valid).
    """
    # Load images
    images = load_patches_parallel(sample["patch_paths"])
    if not images:
        return {"sample_id": sample["sample_id"], "error": "no_images", "raw_output": ""}

    prompt = sample["prompt"]

    # Build message in MedGemma chat format
    content = [{"type": "image", "image": img} for img in images]
    content.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": content}]

    try:
        # Tokenise
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )
        # Move to device with correct dtypes
        inputs = {
            k: v.to(model.device, dtype=torch.bfloat16) if v.is_floating_point()
            else v.to(model.device)
            for k, v in inputs.items()
            if torch.is_tensor(v)
        }

        input_len = inputs["input_ids"].shape[-1]

        # Generate (greedy, deterministic)
        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=True,
            )

        # Decode only the generated tokens
        generated = output_ids[0][input_len:]
        raw_output = processor.decode(generated, skip_special_tokens=True).strip()

        # Parse JSON
        parsed_json = None
        try:
            clean = raw_output
            if "```json" in clean:
                clean = clean.split("```json")[1].split("```")[0]
            elif "```" in clean:
                clean = clean.split("```")[1].split("```")[0]
            start = clean.find("{")
            end = clean.rfind("}") + 1
            if start != -1 and end > start:
                parsed_json = normalize_prediction_keys(json.loads(clean[start:end]))
        except (json.JSONDecodeError, IndexError):
            pass

        return {
            "sample_id": sample["sample_id"],
            "patient_id": sample.get("patient_id", ""),
            "raw_output": raw_output,
            "parsed_json": parsed_json,
            "json_valid": parsed_json is not None,
            "input_tokens": input_len,
            "output_tokens": len(generated),
            "n_images": len(images),
            "error": None,
        }

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return {"sample_id": sample["sample_id"], "error": "OOM", "raw_output": ""}
    except Exception as e:
        return {"sample_id": sample["sample_id"], "error": str(e), "raw_output": ""}


# Quick single-sample test
if eval_samples:
    print("Running single-sample test...")
    test_result = run_zero_shot_inference(eval_samples[0], model, processor)
    print(f"   JSON valid: {test_result['json_valid']}")
    print(f"   Output preview: {test_result['raw_output'][:200]}")
    if test_result['parsed_json']:
        print(f"   Keys: {list(test_result['parsed_json'].keys())}")

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Running single-sample test...
   JSON valid: True
   Output preview: ```json
{
  "CD274_RNA_proxy_level": "high",
  "MSI_status": "MSI-H",
  "MSI_probability": "0.99",
  "TIL_fraction": "0.1",
  "TME_subtype": "IE",
  "TME_subtype_confidence": "0.85",
  "Immune_phenoty
   Keys: ['cd274_expression', 'msi_status', 'msi_probability', 'til_fraction', 'tme_subtype', 'prediction_entropy', 'immune_phenotype', 'cd8_infiltration', 'immune_score']


## Run Zero-Shot on All Eval Samples

In [10]:
print(f"\n{'=' * 60}")
print(f"Running zero-shot baseline on {len(eval_samples)} samples")
print(f"{'=' * 60}")

all_predictions = []
inference_times = []

for i, sample in enumerate(tqdm(eval_samples, desc="Zero-shot inference")):
    start_time = time.time()
    result = run_zero_shot_inference(sample, model, processor)
    elapsed = time.time() - start_time
    inference_times.append(elapsed)

    result["inference_time_s"] = round(elapsed, 2)
    all_predictions.append(result)

    # Clear cache periodically
    if i % 5 == 0 and torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()

    # Progress update every 10 samples
    if (i + 1) % 10 == 0:
        valid = sum(1 for p in all_predictions if p["json_valid"])
        print(f"   [{i+1}/{len(eval_samples)}] JSON parse rate: {valid}/{i+1} "
              f"({valid/(i+1)*100:.0f}%), avg time: {np.mean(inference_times):.1f}s")

# Summary
n_valid = sum(1 for p in all_predictions if p["json_valid"])
n_errors = sum(1 for p in all_predictions if p.get("error"))
print(f"\nInference complete:")
print(f"   Total:       {len(all_predictions)}")
print(f"   JSON valid:  {n_valid}/{len(all_predictions)} ({n_valid/max(1,len(all_predictions))*100:.0f}%)")
print(f"   Errors:      {n_errors}")
print(f"   Avg time:    {np.mean(inference_times):.1f}s per sample")


Running zero-shot baseline on 188 samples


   [10/188] JSON parse rate: 9/10 (90%), avg time: 27.1s
   [20/188] JSON parse rate: 19/20 (95%), avg time: 24.8s
   [30/188] JSON parse rate: 27/30 (90%), avg time: 27.1s
   [40/188] JSON parse rate: 36/40 (90%), avg time: 27.6s
   [50/188] JSON parse rate: 44/50 (88%), avg time: 28.5s
   [60/188] JSON parse rate: 53/60 (88%), avg time: 29.6s
   [70/188] JSON parse rate: 63/70 (90%), avg time: 28.7s
   [80/188] JSON parse rate: 73/80 (91%), avg time: 28.0s
   [90/188] JSON parse rate: 81/90 (90%), avg time: 28.5s
   [100/188] JSON parse rate: 91/100 (91%), avg time: 27.9s
   [110/188] JSON parse rate: 101/110 (92%), avg time: 27.5s
   [120/188] JSON parse rate: 109/120 (91%), avg time: 27.9s
   [130/188] JSON parse rate: 119/130 (92%), avg time: 27.5s
   [140/188] JSON parse rate: 128/140 (91%), avg time: 27.6s
   [150/188] JSON parse rate: 138/150 (92%), avg time: 27.3s
   [160/188] JSON parse rate: 148/160 (92%), avg time: 27.0s
   [170/188] JSON parse rate: 158/170 (93%), avg time

## Save Raw Predictions

In [7]:
import os
import json

predictions_path = f"{RESULTS_DIR}/zero_shot_predictions.jsonl"

assert os.path.exists(predictions_path), "Predictions file does not exist."

all_predictions = []

with open(predictions_path, "r") as f:
    for line in f:
        line = line.strip()
        if line:
            all_predictions.append(json.loads(line))

print(f"Loaded {len(all_predictions)} predictions from disk")


Loaded 188 predictions from disk


## Compute Zero-Shot Metrics

In [5]:
!pip install -q --upgrade numpy scikit-learn scipy

In [8]:
predictions_path = f"{RESULTS_DIR}/zero_shot_predictions.jsonl"
all_predictions = []
with open(predictions_path) as f:
    for line in f:
        pred = json.loads(line.strip())
        if pred.get("parsed_json"):
            pred["parsed_json"] = normalize_prediction_keys(pred["parsed_json"])
            pred["json_valid"] = True
        all_predictions.append(pred)

n_valid = sum(1 for p in all_predictions if p.get("json_valid"))
print(f"Reloaded {len(all_predictions)} predictions")
print(f"   JSON valid: {n_valid}/{len(all_predictions)}")

# Verify keys
for p in all_predictions:
    if p.get("parsed_json"):
        print(f"   Sample keys: {list(p['parsed_json'].keys())}")
        print(f"   cd274_expression present: {'cd274_expression' in p['parsed_json']}")
        print(f"   msi_status present: {'msi_status' in p['parsed_json']}")
        print(f"   tme_subtype present: {'tme_subtype' in p['parsed_json']}")
        break

Reloaded 188 predictions
   JSON valid: 176/188
   Sample keys: ['cd274_expression', 'msi_status', 'msi_probability', 'til_fraction', 'tme_subtype', 'prediction_entropy', 'immune_phenotype', 'cd8_infiltration', 'immune_score']
   cd274_expression present: True
   msi_status present: True
   tme_subtype present: True


In [9]:
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, mean_absolute_error
from scipy.stats import spearmanr

# Pre-build lookup for O(1) ground truth access (instead of O(n) linear scan)
eval_sample_lookup = {s["sample_id"]: s for s in eval_samples}

def compute_zero_shot_metrics(predictions: list, ground_truth: pd.DataFrame) -> dict:
    metrics = {
        "n_total": len(predictions),
        "n_json_valid": sum(1 for p in predictions if p["json_valid"]),
        "json_parse_rate": 0.0,
        "cd274": {}, "msi": {}, "tme": {}, "til": {}, "immune_score": {},
    }
    metrics["json_parse_rate"] = metrics["n_json_valid"] / max(1, metrics["n_total"])
    valid_preds = [p for p in predictions if p["json_valid"] and p["parsed_json"]]

    # CD274 AUC
    cd274_true, cd274_pred = [], []
    for p in valid_preds:
        sid = p["sample_id"]
        if sid in ground_truth.index and "cd274_expression" in ground_truth.columns:
            true_label = str(ground_truth.loc[sid, "cd274_expression"]).lower()
            pred_label = str(p["parsed_json"].get("cd274_expression", "unknown")).lower()
            if true_label in ("high", "low") and pred_label in ("high", "low"):
                cd274_true.append(1 if true_label == "high" else 0)
                cd274_pred.append(1 if pred_label == "high" else 0)
    if len(cd274_true) >= 2 and len(set(cd274_true)) > 1:
        metrics["cd274"]["auc"] = round(roc_auc_score(cd274_true, cd274_pred), 4)
        metrics["cd274"]["accuracy"] = round(accuracy_score(cd274_true, cd274_pred), 4)
        metrics["cd274"]["f1"] = round(f1_score(cd274_true, cd274_pred, zero_division=0), 4)
    metrics["cd274"]["n_samples"] = len(cd274_true)

    # MSI Accuracy (O(1) lookup)
    msi_true, msi_pred = [], []
    for p in valid_preds:
        sid = p["sample_id"]
        pj = p["parsed_json"]
        pred_msi = str(pj.get("msi_status", "unknown")).upper()
        gt_sample = eval_sample_lookup.get(sid)
        gt_resp = None
        if gt_sample:
            try:
                gt_resp = json.loads(gt_sample["response"])
            except Exception:
                pass
        if gt_resp and gt_resp.get("msi_status") not in (None, "unknown"):
            true_msi = gt_resp["msi_status"].upper()
            if true_msi in ("MSI-H", "MSS") and pred_msi in ("MSI-H", "MSS"):
                msi_true.append(1 if true_msi == "MSI-H" else 0)
                msi_pred.append(1 if pred_msi == "MSI-H" else 0)
    if len(msi_true) >= 2 and len(set(msi_true)) > 1:
        metrics["msi"]["auc"] = round(roc_auc_score(msi_true, msi_pred), 4)
        metrics["msi"]["accuracy"] = round(accuracy_score(msi_true, msi_pred), 4)
    elif msi_true:
        metrics["msi"]["accuracy"] = round(accuracy_score(msi_true, msi_pred), 4)
    metrics["msi"]["n_samples"] = len(msi_true)

    # TME Subtype Accuracy (O(1) lookup)
    tme_labels = {"IE": 0, "IE/F": 1, "F": 2, "D": 3}
    tme_true, tme_pred = [], []
    for p in valid_preds:
        sid = p["sample_id"]
        pj = p["parsed_json"]
        pred_tme = str(pj.get("tme_subtype", "unknown"))
        pred_tme = pred_tme.replace("IE_F", "IE/F").replace("IE-F", "IE/F")
        gt_sample = eval_sample_lookup.get(sid)
        gt_resp = None
        if gt_sample:
            try:
                gt_resp = json.loads(gt_sample["response"])
            except Exception:
                pass
        if gt_resp and gt_resp.get("tme_subtype") in tme_labels and pred_tme in tme_labels:
            tme_true.append(tme_labels[gt_resp["tme_subtype"]])
            tme_pred.append(tme_labels[pred_tme])
    if tme_true:
        metrics["tme"]["accuracy"] = round(accuracy_score(tme_true, tme_pred), 4)
        metrics["tme"]["macro_f1"] = round(f1_score(tme_true, tme_pred, average="macro", zero_division=0), 4)
    metrics["tme"]["n_samples"] = len(tme_true)

    # TIL Fraction Correlation
    til_true, til_pred = [], []
    for p in valid_preds:
        sid = p["sample_id"]
        pj = p["parsed_json"]
        pred_til = pj.get("til_fraction")
        if pred_til is not None and sid in ground_truth.index:
            try:
                pred_val = float(pred_til)
                true_val = float(ground_truth.loc[sid, "immune_score"])
                til_true.append(true_val)
                til_pred.append(pred_val)
            except (ValueError, TypeError):
                pass
    if len(til_true) >= 3:
        rho, pval = spearmanr(til_true, til_pred)
        metrics["til"]["spearman_rho"] = round(rho, 4)
        metrics["til"]["spearman_p"] = round(pval, 6)
        metrics["til"]["mae"] = round(mean_absolute_error(til_true, til_pred), 4)
    metrics["til"]["n_samples"] = len(til_true)

    # Immune Score
    iscore_true, iscore_pred = [], []
    for p in valid_preds:
        sid = p["sample_id"]
        pj = p["parsed_json"]
        pred_is = pj.get("immune_score")
        if pred_is is not None and sid in ground_truth.index:
            try:
                pred_val = float(pred_is)
                true_val = float(ground_truth.loc[sid, "immune_score"])
                iscore_true.append(true_val)
                iscore_pred.append(pred_val)
            except (ValueError, TypeError):
                pass
    if len(iscore_true) >= 3:
        rho, pval = spearmanr(iscore_true, iscore_pred)
        metrics["immune_score"]["spearman_rho"] = round(rho, 4)
        metrics["immune_score"]["mae"] = round(mean_absolute_error(iscore_true, iscore_pred), 4)
    metrics["immune_score"]["n_samples"] = len(iscore_true)

    return metrics


metrics = compute_zero_shot_metrics(all_predictions, ground_truth_df)

print("=" * 60)
print("ZERO-SHOT BASELINE METRICS")
print("=" * 60)
for task, task_metrics in metrics.items():
    if isinstance(task_metrics, dict):
        print(f"\n  {task}:")
        for k, v in task_metrics.items():
            print(f"    {k}: {v}")
    else:
        print(f"  {task}: {task_metrics}")


ZERO-SHOT BASELINE METRICS
  n_total: 188
  n_json_valid: 176
  json_parse_rate: 0.9361702127659575

  cd274:
    auc: 0.5015
    accuracy: 0.4932
    f1: 0.5714
    n_samples: 148

  msi:
    auc: 0.6571
    accuracy: 0.3239
    n_samples: 142

  tme:
    accuracy: 0.2254
    macro_f1: 0.0925
    n_samples: 142

  til:
    spearman_rho: 0.0063
    spearman_p: 0.942289
    mae: 0.3821
    n_samples: 133

  immune_score:
    spearman_rho: -0.137
    mae: 0.2083
    n_samples: 144


## Save Metrics + Report

In [16]:
from datetime import datetime

# Extract inference times from saved predictions (works without re-running the loop)
inference_times = [p["inference_time_s"] for p in all_predictions if "inference_time_s" in p]
avg_time = round(np.mean(inference_times), 2) if inference_times else 0
total_time_min = round(sum(inference_times) / 60, 1) if inference_times else 0

# Handle case where model wasn't loaded in this session
MODEL_ID = globals().get("MODEL_ID", "google/medgemma-1.5-4b-it")
attn_impl = globals().get("attn_impl", "flash_attention_2")

# Save metrics JSON
metrics_path = f"{RESULTS_DIR}/zero_shot_metrics.json"
full_report = {
    "phase": 4,
    "timestamp": datetime.now().isoformat(),
    "model_id": MODEL_ID,
    "attention": attn_impl,
    "n_eval_samples": len(all_predictions),
    "avg_inference_time_s": avg_time,
    "total_inference_time_min": total_time_min,
    "metrics": metrics,
}
with open(metrics_path, 'w') as f:
    json.dump(full_report, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(f"   Avg inference time: {avg_time}s per sample")
print(f"   Total inference time: {total_time_min} min")

# Generate markdown report
report_md = f"""# Phase 4 — Zero-Shot Baseline Report

**Model:** {MODEL_ID}
**Date:** {datetime.now().strftime('%Y-%m-%d')}
**Eval samples:** {len(all_predictions)}
**Avg inference time:** {avg_time}s per sample
**Total inference time:** {total_time_min} min

## Key Metrics

| Task | Metric | Value |
|------|--------|-------|
| JSON Parse Rate | % valid | {metrics['json_parse_rate']*100:.0f}% |
| CD274 (PD-L1 proxy) | AUC | {metrics['cd274'].get('auc', 'N/A')} |
| CD274 | Accuracy | {metrics['cd274'].get('accuracy', 'N/A')} |
| MSI Status | Accuracy | {metrics['msi'].get('accuracy', 'N/A')} |
| TME Subtype | Accuracy | {metrics['tme'].get('accuracy', 'N/A')} |
| TIL Fraction | Spearman ρ | {metrics['til'].get('spearman_rho', 'N/A')} |
| Immune Score | Spearman ρ | {metrics['immune_score'].get('spearman_rho', 'N/A')} |

## Notes
- Zero-shot = no fine-tuning, no adapters
- Metrics computed on test + val combined ({len(all_predictions)} samples)
- These are the BASELINE to beat in Phase 5 (fine-tuning)

## Next Steps
- Phase 5: Fine-tune with DoRA (r=16, alpha=32)
- Expected improvements: CD274 AUC +10-20%, TME accuracy +15-25%
"""

report_path = f"{RESULTS_DIR}/zero_shot_report.md"
with open(report_path, 'w') as f:
    f.write(report_md)
print(f"Report saved to {report_path}")

Metrics saved to /content/drive/MyDrive/ImmunoPath/results/phase4/zero_shot_metrics.json
   Avg inference time: 26.4s per sample
   Total inference time: 82.7 min
Report saved to /content/drive/MyDrive/ImmunoPath/results/phase4/zero_shot_report.md


## Phase 4 Summary

In [17]:
print("\n" + "=" * 60)
print("PHASE 4 — ZERO-SHOT BASELINE COMPLETE")
print("=" * 60)
print(f"\n  Model: {MODEL_ID}")
print(f"  Samples evaluated: {len(all_predictions)}")
print(f"  JSON parse rate:   {metrics['json_parse_rate']*100:.0f}%")
print(f"  Avg inference time: {avg_time}s per sample ({total_time_min} min total)")
print(f"  CD274 AUC:         {metrics['cd274'].get('auc', 'N/A')}")
print(f"  MSI accuracy:      {metrics['msi'].get('accuracy', 'N/A')}")
print(f"  TME accuracy:      {metrics['tme'].get('accuracy', 'N/A')}")
print(f"  TIL Spearman ρ:    {metrics['til'].get('spearman_rho', 'N/A')}")

print(f"\n{'=' * 60}")
print("NEXT: Phase 5 — Fine-Tuning MedGemma with DoRA")
print(f"{'=' * 60}")


PHASE 4 — ZERO-SHOT BASELINE COMPLETE

  Model: google/medgemma-1.5-4b-it
  Samples evaluated: 188
  JSON parse rate:   94%
  Avg inference time: 26.4s per sample (82.7 min total)
  CD274 AUC:         0.5015
  MSI accuracy:      0.3239
  TME accuracy:      0.2254
  TIL Spearman ρ:    0.0063

NEXT: Phase 5 — Fine-Tuning MedGemma with DoRA
